# 🌸 Analisis Berita: TF-IDF vs Skip-gram

Notebook ini membandingkan dua cara merepresentasikan berita menjadi vektor, yaitu **TF-IDF** dan **Skip-gram**, lalu menggunakannya untuk klasifikasi berita **Sport** dan **Finance** dengan **Naive Bayes**.

Alur pengerjaan:

1. Membaca Data dan Pengaturan Penyimpanan Hasil
2. Preprocessing
3. Representasi TF-IDF
4. Skip-gram
5. Eksperimen `vector_size` dan `window` dengan GridSearchCV
6. Perbandingan klasifikasi TF-IDF vs Skip-gram
7. Eksperimen tanda baca dihapus vs tidak dihapus
8. Menyimpan model
9. Aplikasi klasifikasi berita dengan Streamlit
10. Kesimpulan

## 1. Membaca Data dan Pengaturan Penyimpanan Hasil

### 1.1 Membaca Data

Dataset hasil crawling berita Sport dan Finance dibaca menggunakan `pandas`. Jumlah baris, kolom, dan distribusi label dicek sekali di awal untuk memastikan data sesuai dengan yang ditargetkan.

In [1]:
import pandas as pd

df = pd.read_csv("dataset_berita.csv")

print("Ukuran dataset:", df.shape)
print("\nData kosong per kolom:")
print(df.isnull().sum())
print("\nJumlah data duplikat:", df.duplicated().sum())
print("\nDistribusi label:")
print(df["label"].value_counts())

df.head()

Ukuran dataset: (200, 3)

Data kosong per kolom:
id            0
isi_berita    0
label         0
dtype: int64

Jumlah data duplikat: 0

Distribusi label:
label
sport      100
finance    100
Name: count, dtype: int64


,id,isi_berita,label
0,1,Jakarta - Putri Kusuma Wardani percaya diri de...,sport
1,2,Jakarta - Mata Derrick Michael Xzavierro berbi...,sport
2,3,Jakarta - Marc Marquez masih harus beradaptasi...,sport
3,4,Jakarta - Momen debut di Asian Games 2026 tak ...,sport
4,5,Jakarta - MotoGP 2026 akan berlanjut ke San Ma...,sport


### 1.2 Pengaturan Penyimpanan Hasil

Proses yang lama (stemming, TF-IDF, Skip-gram, dan GridSearchCV) hanya perlu dijalankan sekali. Hasilnya disimpan ke dalam folder `FOLDER`. Saat notebook dijalankan ulang, setiap tahap akan **mengecek filenya terlebih dahulu**: jika file ada, hasil langsung dimuat; jika belum ada, tahap tersebut dijalankan lalu hasilnya disimpan.

Pengaturan:

- `FOLDER`: lokasi penyimpanan hasil. Dapat diganti, misalnya `r"C:\Users\acer\Downloads\hasil"`.
- `FORCE_ULANG`: jika diubah menjadi `True`, semua tahap dijalankan ulang dan file lama ditimpa.

Cache juga menyimpan sidik (hash) dari teks hasil preprocessing dan parameter GridSearch. Jika teks atau parameternya berubah, tahap yang bergantung padanya otomatis dihitung ulang sehingga hasil lama yang sudah tidak sesuai tidak ikut terpakai.

File yang dihasilkan:

| File | Isi |
|---|---|
| `hasil_preprocessing.csv` | Teks setelah preprocessing |
| `tfidf.joblib` dan `hasil_tfidf.csv` | Vectorizer, matriks TF-IDF, dan tabelnya |
| `skipgram_demo.joblib` dan `hasil_skipgram.csv` | Skip-gram `vector_size=100` dan tabel 101 kolom |
| `gridsearch_skipgram.joblib` dan `gridsearch_tfidf.joblib` | Hasil GridSearchCV |
| `hasil_preprocessing_tanda_baca.csv` | Teks versi tanda baca dipertahankan |
| `hasil_eksperimen_tanda_baca.joblib` dan `.csv` | Tabel perbandingan tanda baca |

File `.joblib` yang berisi kelas `SkipgramVectorizer` hanya dapat dimuat di notebook ini, karena kelasnya didefinisikan di sini. Model untuk aplikasi (`skipgram.model` dan `naive_bayes.joblib`) disimpan terpisah pada bagian 8.

In [2]:
import os
import joblib

FOLDER = "hasil"        # contoh lain: r"C:\Users\acer\Downloads\hasil"
FORCE_ULANG = False     # True = abaikan file yang sudah ada dan hitung ulang semuanya

os.makedirs(FOLDER, exist_ok=True)


def jalur(nama):
    return os.path.join(FOLDER, nama)


def ada(nama):
    return os.path.exists(jalur(nama)) and not FORCE_ULANG


def muat_cache(nama, **syarat):
    """Mengembalikan isi file jika ada dan semua syarat (sidik, parameter) cocok.
    Jika tidak, mengembalikan None."""
    if not ada(nama):
        return None
    try:
        isi = joblib.load(jalur(nama))
    except Exception:
        return None
    for kunci, nilai in syarat.items():
        if isi.get(kunci) != nilai:
            return None
    return isi


def simpan_cache(nama, **isi):
    joblib.dump(isi, jalur(nama))

## 2. Preprocessing

Preprocessing dilakukan dalam satu fungsi dengan bantuan library `Sastrawi`, yang menyediakan daftar stopword dan stemmer Bahasa Indonesia sehingga tidak perlu library tambahan.

Tahapan di dalam fungsi `preprocess_teks`:

1. Mengubah teks menjadi huruf kecil (case folding).
2. Menghapus URL, angka, dan tanda baca.
3. Menghapus stopword.
4. Stemming, yaitu mengubah kata menjadi bentuk dasarnya.

Hasilnya disimpan pada kolom `teks_preprocessed` dan pada file `hasil_preprocessing.csv`. Stemming membutuhkan waktu beberapa saat, sehingga pada proses berikutnya hasilnya dimuat dari file. Jika isi fungsi `preprocess_teks` diubah, jalankan sekali dengan `FORCE_ULANG = True` atau hapus file `hasil_preprocessing.csv`.

In [3]:
%pip install Sastrawi

Note: you may need to restart the kernel to use updated packages.


In [4]:
import re
import hashlib
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stemmer = StemmerFactory().create_stemmer()
stopword = set(StopWordRemoverFactory().get_stop_words())


def preprocess_teks(teks):
    teks = teks.lower()                          # case folding
    teks = re.sub(r"http\S+|www\S+", " ", teks)  # hapus URL
    teks = re.sub(r"[^a-z\s]", " ", teks)        # hapus angka dan tanda baca
    teks = re.sub(r"\s+", " ", teks).strip()
    tokens = [k for k in teks.split() if k not in stopword]   # hapus stopword
    return stemmer.stem(" ".join(tokens))                      # stemming


def muat_preprocessing():
    if not ada("hasil_preprocessing.csv"):
        return False
    d = pd.read_csv(jalur("hasil_preprocessing.csv"), keep_default_na=False)
    if len(d) == len(df) and (d["id"].values == df["id"].values).all():
        df["teks_preprocessed"] = d["teks_preprocessed"].values
        return True
    return False


if muat_preprocessing():
    print("Hasil preprocessing dimuat dari file (stemming tidak perlu diulang).")
else:
    print("File belum ada, preprocessing dijalankan (stemming membutuhkan waktu)...")
    df["teks_preprocessed"] = df["isi_berita"].apply(preprocess_teks)
    df[["id", "label", "teks_preprocessed"]].to_csv(
        jalur("hasil_preprocessing.csv"), index=False, encoding="utf-8-sig"
    )
    print("Hasil preprocessing disimpan:", jalur("hasil_preprocessing.csv"))

# Sidik teks: dipakai untuk memastikan cache tahap berikutnya masih sesuai
SIDIK = hashlib.md5("||".join(df["teks_preprocessed"]).encode("utf-8")).hexdigest()

print("Jumlah stopword:", len(stopword))
df[["isi_berita", "teks_preprocessed"]].head()

File belum ada, preprocessing dijalankan (stemming membutuhkan waktu)...
Hasil preprocessing disimpan: hasil\hasil_preprocessing.csv
Jumlah stopword: 123


,isi_berita,teks_preprocessed
0,Jakarta - Putri Kusuma Wardani percaya diri de...,jakarta putri kusuma wardani percaya diri komp...
1,Jakarta - Mata Derrick Michael Xzavierro berbi...,jakarta mata derrick michael xzavierro binar b...
2,Jakarta - Marc Marquez masih harus beradaptasi...,jakarta marc marquez adaptasi kondisi lengan k...
3,Jakarta - Momen debut di Asian Games 2026 tak ...,jakarta momen debut asi games tak mau sia sia ...
4,Jakarta - MotoGP 2026 akan berlanjut ke San Ma...,jakarta motogp lanjut san marino seri musim ge...


## 3. Representasi TF-IDF

Setiap berita direpresentasikan dalam bentuk bobot TF-IDF menggunakan `TfidfVectorizer`. Setiap kata unik menjadi satu kolom, sehingga jumlah kolom sama dengan jumlah kata unik pada seluruh berita.

Nilai TF-IDF dihitung dengan:

$$
TFIDF(t,d) = TF(t,d) \times IDF(t)
$$

dengan:

- $TF(t,d)$ adalah frekuensi kemunculan kata $t$ pada dokumen $d$.
- $IDF(t)$ menunjukkan seberapa jarang kata tersebut muncul pada seluruh dokumen.
- $TFIDF(t,d)$ adalah bobot akhir kata $t$ pada dokumen $d$.

Vectorizer dan matriks hasilnya disimpan pada `tfidf.joblib`, sedangkan tabel lengkapnya pada `hasil_tfidf.csv`.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

cache = muat_cache("tfidf.joblib", sidik=SIDIK)

if cache:
    tfidf_vectorizer = cache["vectorizer"]
    X_tfidf = cache["X"]
    print("Hasil TF-IDF dimuat dari file.")
else:
    print("File belum ada, TF-IDF dihitung...")
    tfidf_vectorizer = TfidfVectorizer()
    X_tfidf = tfidf_vectorizer.fit_transform(df["teks_preprocessed"])
    simpan_cache("tfidf.joblib", sidik=SIDIK, vectorizer=tfidf_vectorizer, X=X_tfidf)

    df_tfidf_lengkap = pd.DataFrame(
        X_tfidf.toarray(),
        columns=tfidf_vectorizer.get_feature_names_out()
    )
    df_tfidf_lengkap.insert(0, "id", df["id"].values)
    df_tfidf_lengkap["label"] = df["label"].values
    df_tfidf_lengkap.to_csv(jalur("hasil_tfidf.csv"), index=False, encoding="utf-8-sig")
    print("Hasil TF-IDF disimpan. Ukuran tabel:", df_tfidf_lengkap.shape)

fitur_tfidf = tfidf_vectorizer.get_feature_names_out()
kata_unik = fitur_tfidf

print("Jumlah berita          :", X_tfidf.shape[0])
print("Jumlah kata unik/kolom :", X_tfidf.shape[1])
print("Ukuran matriks TF-IDF  :", X_tfidf.shape)

# Contoh: 5 berita pertama dan 10 kolom pertama
df_tfidf_contoh = pd.DataFrame(X_tfidf[:5, :10].toarray(), columns=fitur_tfidf[:10])
df_tfidf_contoh.insert(0, "id", df["id"].head(5).values)
df_tfidf_contoh["label"] = df["label"].head(5).values
display(df_tfidf_contoh)

Hasil TF-IDF dimuat dari file.
Jumlah berita          : 200
Jumlah kata unik/kolom : 5271
Ukuran matriks TF-IDF  : (200, 5271)


,id,aam,aan,abadi,abai,abal,abang,abdukhodir,abdul,abdullah,aberdeen,label
0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sport
1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sport
2,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sport
3,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sport
4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sport


## 4. Skip-gram

### 4.1 Menyiapkan Kelas Skip-gram

Skip-gram merupakan salah satu arsitektur Word2Vec. Setiap kata diubah menjadi vektor dengan jumlah dimensi sebesar `vector_size`. Satu berita kemudian direpresentasikan sebagai **rata-rata vektor kata-kata** di dalamnya. Dengan cara ini, jumlah kolom ditentukan oleh `vector_size`, bukan lagi oleh jumlah kata unik seperti pada TF-IDF.

Kelas `SkipgramVectorizer` membungkus Word2Vec agar dapat dipakai bersama `Pipeline` dan `GridSearchCV`. Skip-gram dilatih ulang di dalam setiap fold sehingga data uji tidak ikut dipelajari.

In [7]:
%pip install gensim

In [9]:
import pandas as pd
import numpy as np
from IPython.display import display

# 1. Latih Skip-gram dan ubah semua berita menjadi vektor
sg = SkipgramVectorizer(vector_size=100, window=5).fit(df["teks_preprocessed"])
X_sg = sg.transform(df["teks_preprocessed"])

# 2. Ukuran hasil
print("Jumlah kata unik (kosakata Skip-gram):", len(sg.w2v_.wv))
print("Ukuran matriks Skip-gram              :", X_sg.shape)
print("Jumlah kolom + label                  :", X_sg.shape[1] + 1)

# 3. Tabel hasil: 100 kolom vektor + label = 101 kolom
df_sg = pd.DataFrame(X_sg, columns=[f"v{i}" for i in range(1, X_sg.shape[1] + 1)])
df_sg["label"] = df["label"].values

print("\nTabel Skip-gram (5 berita pertama, 10 kolom pertama + label):")
display(df_sg.iloc[:5, list(range(10)) + [-1]])

# 4. Vektor satu kata (ganti 'motogp' dengan kata lain yang ada di datamu)
kata = "motogp"
print(f"\nVektor kata '{kata}' (10 dimensi pertama dari {sg.vector_size}):")
print(sg.w2v_.wv[kata][:10])

# 5. Kata-kata yang paling mirip
print(f"\n5 kata paling mirip dengan '{kata}':")
for k, skor in sg.w2v_.wv.most_similar(kata, topn=5):
    print(f"  {k:<15} {skor:.4f}")

# 6. Rata-rata vektor per label (untuk melihat sport dan finance berbeda atau tidak)
print("\nRata-rata 5 dimensi pertama per label:")
display(df_sg.groupby("label")[["v1", "v2", "v3", "v4", "v5"]].mean().round(4))

Jumlah kata unik (kosakata Skip-gram): 5295
Ukuran matriks Skip-gram              : (200, 100)
Jumlah kolom + label                  : 101

Tabel Skip-gram (5 berita pertama, 10 kolom pertama + label):


,v1,v2,v3,v4,v5,v6,v7,v8,v9,v10,label
0,-0.096291,-0.274827,-0.034285,0.022910,-0.453925,0.076065,-0.225470,0.336979,-0.382457,0.194938,sport
1,-0.176647,-0.399241,-0.161397,0.104222,-0.208327,0.141438,-0.185062,0.411890,-0.271371,0.206097,sport
2,-0.214362,-0.351240,-0.305462,0.206170,-0.163492,0.257289,-0.242573,0.044747,-0.200680,0.270858,sport
3,-0.042399,-0.250380,-0.214366,0.113532,-0.326513,0.163412,-0.351423,0.282266,-0.360040,0.316177,sport
4,-0.065102,-0.191482,-0.271183,0.219342,0.060556,0.152387,-0.238858,0.040903,-0.156779,0.394961,sport



Vektor kata 'motogp' (10 dimensi pertama dari 100):
[-0.30777165 -0.12617154  0.33924177  0.3918607   0.09315474  0.1901813
 -0.83027923 -0.05250595 -0.36514485  0.4778219 ]

5 kata paling mirip dengan 'motogp':
  san             0.7193
  warta           0.7116
  merupaya        0.7028
  home            0.6921
  aldeguer        0.6839

Rata-rata 5 dimensi pertama per label:


,v1,v2,v3,v4,v5
label,,,,,
finance,0.0733,-0.210,-0.2575,0.2072,-0.1673
sport,0.0032,-0.418,-0.2289,0.1379,-0.1316


### 4.2 Melatih Skip-gram dan Membandingkan Jumlah Kata Unik

Skip-gram dilatih dengan `vector_size=100` dan `window=5` sebagai pengaturan awal. Jumlah kata unik sebelum Skip-gram (dari TF-IDF) dibandingkan dengan jumlah kata pada kosakata Skip-gram.

Selisih yang kecil masih wajar karena `TfidfVectorizer` membuang token satu huruf, sedangkan Word2Vec tidak. Jika `min_count` diperbesar, kata yang jarang muncul akan terbuang dan jumlah kosakata menurun.

In [10]:
jumlah_kata_sebelum = len(kata_unik)

corpus = [teks.split() for teks in df["teks_preprocessed"]]

w2v = Word2Vec(
    sentences=corpus,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,
    epochs=50,
    seed=42,
    workers=1
)

jumlah_kata_sesudah = len(w2v.wv)

print("Kata unik sebelum Skip-gram :", jumlah_kata_sebelum)
print("Kata unik sesudah Skip-gram :", jumlah_kata_sesudah)

Kata unik sebelum Skip-gram : 5271
Kata unik sesudah Skip-gram : 5295


### 4.3 Perbandingan Jumlah Kolom TF-IDF dan Skip-gram

Pada TF-IDF, setiap kata unik menjadi satu kolom. Pada Skip-gram, jumlah kolom hanya sebanyak `vector_size`. Setelah label ditambahkan sebagai satu kolom, tabel Skip-gram dengan `vector_size=100` memiliki 101 kolom. Hasilnya disimpan pada `skipgram_demo.joblib`.

In [11]:
cache = muat_cache("skipgram_demo.joblib", sidik=SIDIK)

if cache:
    sg_vec = cache["sg_vec"]
    X_sg = cache["X_sg"]
    print("Hasil Skip-gram dimuat dari file.")
else:
    print("File belum ada, Skip-gram dilatih...")
    sg_vec = SkipgramVectorizer(vector_size=100, window=5).fit(df["teks_preprocessed"])
    X_sg = sg_vec.transform(df["teks_preprocessed"])
    simpan_cache("skipgram_demo.joblib", sidik=SIDIK, sg_vec=sg_vec, X_sg=X_sg)
    print("Hasil Skip-gram disimpan.")

ringkasan = pd.DataFrame({
    "Representasi": ["TF-IDF", "Skip-gram"],
    "Kata unik": [jumlah_kata_sebelum, len(sg_vec.w2v_.wv)],
    "Jumlah kolom fitur": [X_tfidf.shape[1], X_sg.shape[1]],
    "Kolom + label": [X_tfidf.shape[1] + 1, X_sg.shape[1] + 1],
    "Ukuran matriks": [str(X_tfidf.shape), str(X_sg.shape)],
})

display(ringkasan)

File belum ada, Skip-gram dilatih...
Hasil Skip-gram disimpan.


,Representasi,Kata unik,Jumlah kolom fitur,Kolom + label,Ukuran matriks
0,TF-IDF,5271,5271,5272,"(200, 5271)"
1,Skip-gram,5295,100,101,"(200, 100)"


### 4.4 Contoh Vektor TF-IDF dan Skip-gram

- **TF-IDF**: satu berita berupa vektor sangat panjang yang sebagian besar berisi nol. Nilainya adalah bobot kata tertentu, sehingga yang ditampilkan hanya 10 kata dengan bobot terbesar.
- **Skip-gram**: satu kata berupa vektor padat dengan jumlah dimensi sebesar `vector_size`. Kata yang konteksnya mirip memiliki vektor yang berdekatan, sehingga dapat dilihat dari daftar kata yang paling mirip.

In [12]:
# ---------- TF-IDF: berita pertama, 10 kata dengan bobot terbesar ----------
baris = X_tfidf[0].toarray().ravel()
top = baris.argsort()[::-1][:10]

df_contoh_tfidf = pd.DataFrame({
    "kata": fitur_tfidf[top],
    "bobot_tfidf": baris[top]
})

print("Vektor TF-IDF berita 1 (dari", X_tfidf.shape[1], "kolom, hanya 10 bobot terbesar):")
display(df_contoh_tfidf)

# ---------- Skip-gram: vektor satu kata ----------
print("\nVektor kata 'motogp' (10 dimensi pertama dari", sg_vec.vector_size, "dimensi):")
print(sg_vec.w2v_.wv["motogp"][:10])

print("\n5 kata paling mirip dengan 'motogp':")
for kata, skor in sg_vec.w2v_.wv.most_similar("motogp", topn=5):
    print(f"  {kata:<15} {skor:.4f}")

Vektor TF-IDF berita 1 (dari 5271 kolom, hanya 10 bobot terbesar):


,kata,bobot_tfidf
0,putri,0.478198
1,asi,0.301109
2,games,0.296511
3,regu,0.187016
4,kusuma,0.167643
5,komposisi,0.155527
6,pbsi,0.146930
7,kw,0.146930
8,kuat,0.129100
9,tim,0.116192



Vektor kata 'motogp' (10 dimensi pertama dari 100 dimensi):
[-0.30777165 -0.12617154  0.33924177  0.3918607   0.09315474  0.1901813
 -0.83027923 -0.05250595 -0.36514485  0.4778219 ]

5 kata paling mirip dengan 'motogp':
  san             0.7193
  warta           0.7116
  merupaya        0.7028
  home            0.6921
  aldeguer        0.6839


In [13]:
# ---------- Skip-gram: tabel seluruh berita (100 kolom + label = 101 kolom) ----------
df_sg = pd.DataFrame(
    X_sg,
    columns=[f"v{i}" for i in range(1, X_sg.shape[1] + 1)]
)
df_sg["label"] = df["label"].values

print("Ukuran tabel Skip-gram:", df_sg.shape)
display(df_sg.head(5))

df_sg.to_csv(jalur("hasil_skipgram.csv"), index=False, encoding="utf-8-sig")
print("Hasil Skip-gram berhasil disimpan:", jalur("hasil_skipgram.csv"))

Ukuran tabel Skip-gram: (200, 101)


,v1,v2,v3,v4,v5,v6,v7,v8,v9,v10,...,v92,v93,v94,v95,v96,v97,v98,v99,v100,label
0,-0.096291,-0.274827,-0.034285,0.022910,-0.453925,0.076065,-0.225470,0.336979,-0.382457,0.194938,...,-0.147770,-0.294563,-0.354166,0.075016,-0.209348,0.050684,-0.086855,-0.177248,0.065978,sport
1,-0.176647,-0.399241,-0.161397,0.104222,-0.208327,0.141438,-0.185062,0.411890,-0.271371,0.206097,...,-0.235781,-0.508681,-0.331654,0.182135,-0.211314,-0.039681,-0.000866,-0.116197,-0.099824,sport
2,-0.214362,-0.351240,-0.305462,0.206170,-0.163492,0.257289,-0.242573,0.044747,-0.200680,0.270858,...,0.185838,-0.143834,-0.441295,0.050830,-0.159262,-0.252788,0.150028,-0.103856,0.025539,sport
3,-0.042399,-0.250380,-0.214366,0.113532,-0.326513,0.163412,-0.351423,0.282266,-0.360040,0.316177,...,-0.147808,-0.429727,-0.345264,0.077524,-0.292179,0.156407,-0.073351,-0.094554,0.087524,sport
4,-0.065102,-0.191482,-0.271183,0.219342,0.060556,0.152387,-0.238858,0.040903,-0.156779,0.394961,...,-0.003701,-0.368871,-0.644878,-0.035083,-0.069309,-0.328501,0.041979,-0.153794,-0.119958,sport


Hasil Skip-gram berhasil disimpan: hasil\hasil_skipgram.csv


## 5. Eksperimen `vector_size` dan `window` dengan GridSearchCV

### 5.1 Membagi Data Latih dan Data Uji

Data dibagi menjadi 80% data latih dan 20% data uji dengan `stratify` agar proporsi sport dan finance tetap seimbang. GridSearchCV hanya memakai data latih. Data uji baru dipakai pada tahap perbandingan akhir.

Karena jumlah data hanya 200 berita, data uji berisi 40 berita. Selisih satu berita saja sudah mengubah akurasi sebesar 2,5%, sehingga akurasi cross validation (CV) lebih layak dijadikan patokan utama.

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.naive_bayes import GaussianNB, MultinomialNB

X = df["teks_preprocessed"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Data latih:", len(X_train))
print("Data uji  :", len(X_test))

Data latih: 160
Data uji  : 40


### 5.2 GridSearchCV untuk Skip-gram + Naive Bayes

Kombinasi yang dicoba:

- `vector_size`: 50, 100, 200, 300
- `window`: 3, 5, 10

Naive Bayes yang dipakai untuk Skip-gram adalah `GaussianNB`, karena vektor Skip-gram memiliki nilai negatif sedangkan `MultinomialNB` hanya menerima nilai non-negatif. Seluruh kombinasi diuji dengan 5-fold cross validation sehingga ada 12 kombinasi × 5 fold = 60 kali pelatihan. Hasilnya disimpan pada `gridsearch_skipgram.joblib`; jika parameter di bawah diubah, GridSearch otomatis dijalankan ulang.

In [15]:
pipe_sg = Pipeline([
    ("sg", SkipgramVectorizer()),
    ("nb", GaussianNB()),
])

param_sg = {
    "sg__vector_size": [50, 100, 200, 300],
    "sg__window": [3, 5, 10],
}

cache = muat_cache("gridsearch_skipgram.joblib", sidik=SIDIK, param=param_sg)

if cache:
    gs_sg = cache["gs"]
    print("Hasil GridSearchCV Skip-gram dimuat dari file.")
else:
    print("File belum ada, GridSearchCV Skip-gram dijalankan...")
    gs_sg = GridSearchCV(pipe_sg, param_sg, cv=cv, scoring="accuracy")
    gs_sg.fit(X_train, y_train)
    simpan_cache("gridsearch_skipgram.joblib", sidik=SIDIK, param=param_sg, gs=gs_sg)
    print("Hasil GridSearchCV Skip-gram disimpan.")

print("Parameter terbaik Skip-gram :", gs_sg.best_params_)
print("Akurasi CV terbaik          :", round(gs_sg.best_score_, 4))

File belum ada, GridSearchCV Skip-gram dijalankan...
Hasil GridSearchCV Skip-gram disimpan.
Parameter terbaik Skip-gram : {'sg__vector_size': 100, 'sg__window': 5}
Akurasi CV terbaik          : 0.975


### 5.3 Tabel Eksperimen `vector_size` dan `window`

Tabel berikut menampilkan rata-rata akurasi cross validation untuk setiap kombinasi. Baris menunjukkan `vector_size` dan kolom menunjukkan `window`. Kombinasi dengan nilai tertinggi menjadi parameter terbaik.

In [16]:
hasil_sg = pd.DataFrame(gs_sg.cv_results_)

tabel_eksperimen = hasil_sg.pivot(
    index="param_sg__vector_size",
    columns="param_sg__window",
    values="mean_test_score"
).round(4)

tabel_eksperimen.index.name = "vector_size"
tabel_eksperimen.columns.name = "window"

display(tabel_eksperimen)

window,3,5,10
vector_size,,,
50,0.9688,0.9625,0.9688
100,0.9625,0.9750,0.9688
200,0.9688,0.9750,0.9688
300,0.9688,0.9688,0.9688


### 5.4 GridSearchCV untuk TF-IDF + Naive Bayes (Pembanding)

TF-IDF tidak memiliki parameter `vector_size` dan `window`. Jumlah kolomnya ditentukan oleh jumlah kata unik. Sebagai pembanding, dicari nilai `alpha` (smoothing) terbaik pada `MultinomialNB` dengan cross validation yang sama.

In [17]:
pipe_tfidf = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("nb", MultinomialNB()),
])

param_tfidf = {"nb__alpha": [0.01, 0.1, 0.5, 1.0]}

cache = muat_cache("gridsearch_tfidf.joblib", sidik=SIDIK, param=param_tfidf)

if cache:
    gs_tfidf = cache["gs"]
    print("Hasil GridSearchCV TF-IDF dimuat dari file.")
else:
    print("File belum ada, GridSearchCV TF-IDF dijalankan...")
    gs_tfidf = GridSearchCV(pipe_tfidf, param_tfidf, cv=cv, scoring="accuracy")
    gs_tfidf.fit(X_train, y_train)
    simpan_cache("gridsearch_tfidf.joblib", sidik=SIDIK, param=param_tfidf, gs=gs_tfidf)
    print("Hasil GridSearchCV TF-IDF disimpan.")

print("Parameter terbaik TF-IDF :", gs_tfidf.best_params_)
print("Akurasi CV terbaik       :", round(gs_tfidf.best_score_, 4))

File belum ada, GridSearchCV TF-IDF dijalankan...
Hasil GridSearchCV TF-IDF disimpan.
Parameter terbaik TF-IDF : {'nb__alpha': 0.01}
Akurasi CV terbaik       : 0.9812


## 6. Perbandingan Klasifikasi TF-IDF dan Skip-gram

Kedua model dengan parameter terbaiknya diuji pada data uji yang belum pernah dilihat. Hasil dibandingkan menggunakan akurasi CV, akurasi data uji, serta `classification_report` (precision, recall, dan f1-score).

Sesuai arahan dosen, jika hasil TF-IDF dan Skip-gram sama atau selisihnya tidak berarti, maka dipilih **Skip-gram**.

In [18]:
from sklearn.metrics import accuracy_score, classification_report

baris = []

for nama, gs in [("TF-IDF + Naive Bayes", gs_tfidf),
                 ("Skip-gram + Naive Bayes", gs_sg)]:

    pred = gs.predict(X_test)

    baris.append({
        "Model": nama,
        "Parameter terbaik": gs.best_params_,
        "Akurasi CV (data latih)": round(gs.best_score_, 4),
        "Akurasi data uji": round(accuracy_score(y_test, pred), 4),
    })

    print("=====", nama, "=====")
    print(classification_report(y_test, pred))

tabel_perbandingan = pd.DataFrame(baris)
display(tabel_perbandingan)

===== TF-IDF + Naive Bayes =====
              precision    recall  f1-score   support

     finance       1.00      0.95      0.97        20
       sport       0.95      1.00      0.98        20

    accuracy                           0.97        40
   macro avg       0.98      0.97      0.97        40
weighted avg       0.98      0.97      0.97        40

===== Skip-gram + Naive Bayes =====
              precision    recall  f1-score   support

     finance       1.00      1.00      1.00        20
       sport       1.00      1.00      1.00        20

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



,Model,Parameter terbaik,Akurasi CV (data latih),Akurasi data uji
0,TF-IDF + Naive Bayes,{'nb__alpha': 0.01},0.9812,0.975
1,Skip-gram + Naive Bayes,"{'sg__vector_size': 100, 'sg__window': 5}",0.9750,1.000


## 7. Eksperimen Tanda Baca: Dihapus vs Tidak Dihapus

Pada bagian ini dibandingkan dua versi preprocessing:

- **Tanda baca dihapus**: seperti pada bagian 2, semua karakter selain huruf dibuang.
- **Tanda baca dipertahankan**: tanda baca dipisahkan dari kata dan diperlakukan sebagai token tersendiri (misalnya `,` `.` `-`). Angka dan URL tetap dihapus pada kedua versi sehingga yang berbeda hanya perlakuan tanda baca.

`TfidfVectorizer` secara bawaan membuang tanda baca melalui `token_pattern`-nya. Agar perbandingan bermakna, digunakan `token_pattern=r"\S+"` pada kedua versi sehingga tanda baca ikut menjadi token jika memang dipertahankan.

### 7.1 Membuat Dua Versi Teks

In [19]:
from functools import lru_cache


@lru_cache(maxsize=None)
def _stem_kata(kata):
    return stemmer.stem(kata)


def preprocess_variasi(teks, hapus_tanda_baca=True):
    """hapus_tanda_baca=True  -> sama dengan preprocess_teks.
    hapus_tanda_baca=False -> tanda baca dipertahankan sebagai token terpisah."""
    if hapus_tanda_baca:
        return preprocess_teks(teks)

    teks = teks.lower()
    teks = re.sub(r"http\S+|www\S+", " ", teks)
    teks = re.sub(r"\d+", " ", teks)
    teks = re.sub(r"([^\w\s])", r" \1 ", teks)   # pisahkan tanda baca dari kata
    teks = re.sub(r"\s+", " ", teks).strip()

    hasil = []
    for tok in teks.split():
        if tok in stopword:
            continue
        # stemming hanya untuk token huruf, tanda baca dibiarkan
        hasil.append(_stem_kata(tok) if tok.isalpha() else tok)
    return " ".join(hasil)


# Versi tanda baca dihapus sama dengan hasil bagian 2, jadi tidak perlu dihitung ulang
df["teks_hapus"] = df["teks_preprocessed"]


def muat_tanda_baca():
    if not ada("hasil_preprocessing_tanda_baca.csv"):
        return False
    d = pd.read_csv(jalur("hasil_preprocessing_tanda_baca.csv"), keep_default_na=False)
    if len(d) == len(df) and (d["teks_hapus"].values == df["teks_hapus"].values).all():
        df["teks_tanpa_hapus"] = d["teks_tanpa_hapus"].values
        return True
    return False


if muat_tanda_baca():
    print("Teks versi tanda baca dipertahankan dimuat dari file.")
else:
    print("File belum ada, teks versi tanda baca dipertahankan dibuat...")
    df["teks_tanpa_hapus"] = df["isi_berita"].apply(
        lambda t: preprocess_variasi(t, hapus_tanda_baca=False)
    )
    df[["id", "teks_hapus", "teks_tanpa_hapus"]].to_csv(
        jalur("hasil_preprocessing_tanda_baca.csv"), index=False, encoding="utf-8-sig"
    )
    print("Disimpan:", jalur("hasil_preprocessing_tanda_baca.csv"))

df[["teks_hapus", "teks_tanpa_hapus"]].head(3)

File belum ada, teks versi tanda baca dipertahankan dibuat...
Disimpan: hasil\hasil_preprocessing_tanda_baca.csv


,teks_hapus,teks_tanpa_hapus
0,jakarta putri kusuma wardani percaya diri komp...,jakarta - putri kusuma wardani percaya diri ko...
1,jakarta mata derrick michael xzavierro binar b...,jakarta - mata derrick michael xzavierro binar...
2,jakarta marc marquez adaptasi kondisi lengan k...,jakarta - marc marquez adaptasi kondisi lengan...


### 7.2 Menjalankan Eksperimen

Pembagian data dan fold cross validation dibuat satu kali dan dipakai bersama oleh kedua versi, sehingga satu-satunya yang berubah adalah perlakuan tanda baca. Setiap versi diuji dengan TF-IDF maupun Skip-gram, lengkap dengan GridSearchCV. Tabel hasilnya disimpan pada `hasil_eksperimen_tanda_baca.joblib` dan `.csv`.

In [20]:
idx_train, idx_test = train_test_split(
    df.index,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

y_tr = df.loc[idx_train, "label"]
y_te = df.loc[idx_test, "label"]

varian = {
    "Tanda baca dihapus": "teks_hapus",
    "Tanda baca dipertahankan": "teks_tanpa_hapus",
}


def jalankan_eksperimen():
    baris = []

    for nama_varian, kolom in varian.items():

        X_tr = df.loc[idx_train, kolom]
        X_te = df.loc[idx_test, kolom]

        # jumlah token unik pada versi ini
        n_vocab = len(
            TfidfVectorizer(token_pattern=r"\S+").fit(df[kolom]).vocabulary_
        )

        # ---------- TF-IDF + MultinomialNB ----------
        gs_t = GridSearchCV(
            Pipeline([
                ("tfidf", TfidfVectorizer(token_pattern=r"\S+")),
                ("nb", MultinomialNB()),
            ]),
            param_tfidf,
            cv=cv,
            scoring="accuracy"
        ).fit(X_tr, y_tr)

        baris.append({
            "Preprocessing": nama_varian,
            "Representasi": "TF-IDF",
            "Token unik": n_vocab,
            "Jumlah kolom": len(gs_t.best_estimator_["tfidf"].vocabulary_),
            "Parameter terbaik": gs_t.best_params_,
            "Akurasi CV": round(gs_t.best_score_, 4),
            "Akurasi data uji": round(accuracy_score(y_te, gs_t.predict(X_te)), 4),
        })

        # ---------- Skip-gram + GaussianNB ----------
        gs_s = GridSearchCV(
            Pipeline([
                ("sg", SkipgramVectorizer()),
                ("nb", GaussianNB()),
            ]),
            param_sg,
            cv=cv,
            scoring="accuracy"
        ).fit(X_tr, y_tr)

        baris.append({
            "Preprocessing": nama_varian,
            "Representasi": "Skip-gram",
            "Token unik": n_vocab,
            "Jumlah kolom": gs_s.best_params_["sg__vector_size"],
            "Parameter terbaik": gs_s.best_params_,
            "Akurasi CV": round(gs_s.best_score_, 4),
            "Akurasi data uji": round(accuracy_score(y_te, gs_s.predict(X_te)), 4),
        })

    return pd.DataFrame(baris)


# Sidik teks versi tanda baca dipertahankan
SIDIK_TB = hashlib.md5("||".join(df["teks_tanpa_hapus"]).encode("utf-8")).hexdigest()

cache = muat_cache(
    "hasil_eksperimen_tanda_baca.joblib",
    sidik=SIDIK, sidik_tb=SIDIK_TB, param_sg=param_sg, param_tfidf=param_tfidf
)

if cache:
    hasil_tanda_baca = cache["hasil"]
    print("Hasil eksperimen tanda baca dimuat dari file.")
else:
    print("File belum ada, eksperimen tanda baca dijalankan...")
    hasil_tanda_baca = jalankan_eksperimen()
    simpan_cache(
        "hasil_eksperimen_tanda_baca.joblib",
        sidik=SIDIK, sidik_tb=SIDIK_TB, param_sg=param_sg, param_tfidf=param_tfidf,
        hasil=hasil_tanda_baca
    )
    hasil_tanda_baca.to_csv(
        jalur("hasil_eksperimen_tanda_baca.csv"), index=False, encoding="utf-8-sig"
    )
    print("Hasil eksperimen disimpan.")

display(hasil_tanda_baca)

File belum ada, eksperimen tanda baca dijalankan...
Hasil eksperimen disimpan.


,Preprocessing,Representasi,Token unik,Jumlah kolom,Parameter terbaik,Akurasi CV,Akurasi data uji
0,Tanda baca dihapus,TF-IDF,5295,4784,{'nb__alpha': 0.01},0.9812,0.975
1,Tanda baca dihapus,Skip-gram,5295,100,"{'sg__vector_size': 100, 'sg__window': 5}",0.9750,1.000
2,Tanda baca dipertahankan,TF-IDF,5335,4824,{'nb__alpha': 0.01},0.9812,0.975
3,Tanda baca dipertahankan,Skip-gram,5335,100,"{'sg__vector_size': 100, 'sg__window': 3}",0.9688,1.000


### 7.3 Cara Membaca Hasil

- Bandingkan dua baris dengan representasi yang sama, yaitu TF-IDF dihapus dengan TF-IDF dipertahankan, dan Skip-gram dihapus dengan Skip-gram dipertahankan.
- Jadikan **Akurasi CV** sebagai patokan utama, karena data uji hanya 40 berita.
- Jika selisihnya kecil (di bawah sekitar 1 sampai 2%), kesimpulannya adalah tanda baca tidak berpengaruh berarti pada dataset ini.
- Kolom **Token unik** menunjukkan bahwa versi yang mempertahankan tanda baca memiliki beberapa token tambahan seperti `,` `.` dan `-`.

Jika ternyata versi **tanda baca dipertahankan** memberikan hasil yang jelas lebih baik, fungsi `preprocess` pada `app.py` (bagian 9) harus disamakan dengan `preprocess_variasi(..., hapus_tanda_baca=False)`, dan model pada bagian 8 dilatih ulang menggunakan teks versi tersebut.

## 8. Menyimpan Model

### 8.1 Menyimpan Skip-gram dan Naive Bayes

Model dilatih ulang dengan parameter terbaik dari bagian 5 menggunakan seluruh 200 berita, kemudian disimpan menjadi dua file terpisah:

- `skipgram.model`: model Skip-gram dalam format bawaan gensim.
- `naive_bayes.joblib`: model Naive Bayes dalam format sklearn (`joblib`).

Kedua komponen disimpan terpisah karena kelas `SkipgramVectorizer` didefinisikan di notebook, sehingga menyimpan seluruh pipeline dengan `joblib` akan gagal saat dibuka kembali di aplikasi Streamlit.

In [21]:
import joblib
from sklearn.base import clone

# Latih ulang dengan parameter terbaik pada seluruh data
model_final = clone(gs_sg.best_estimator_).fit(X, y)

sg_final = model_final.named_steps["sg"]
nb_final = model_final.named_steps["nb"]

# Simpan terpisah
sg_final.w2v_.save("skipgram.model")
joblib.dump(nb_final, "naive_bayes.joblib")

print("Parameter terbaik :", gs_sg.best_params_)
print("Tersimpan         : skipgram.model dan naive_bayes.joblib")

Parameter terbaik : {'sg__vector_size': 100, 'sg__window': 5}
Tersimpan         : skipgram.model dan naive_bayes.joblib


### 8.2 Menguji Model yang Disimpan

Model dimuat kembali dari file untuk memastikan alurnya berjalan: teks masuk, diproses, diubah menjadi vektor Skip-gram, lalu vektor tersebut diklasifikasikan oleh Naive Bayes.

In [22]:
w2v_load = Word2Vec.load("skipgram.model")
nb_load = joblib.load("naive_bayes.joblib")


def prediksi(teks):
    bersih = preprocess_teks(teks)
    vec = [w2v_load.wv[k] for k in bersih.split() if k in w2v_load.wv]
    vec = np.mean(vec, axis=0) if vec else np.zeros(w2v_load.vector_size)
    return nb_load.predict([vec])[0]


print(prediksi("Harga emas Antam naik, rupiah menguat terhadap dolar"))
print(prediksi("Marc Marquez juara MotoGP setelah balapan sengit"))

finance
sport
